# 综合项目：螺旋数据分类

## 学习目标

从数据探索到训练、验证和决策边界可视化，完成一个离线可复现的 NumPy MLP 项目。

## 概念模型

二维螺旋无法被单条直线分开，适合验证隐藏层和非线性的价值。完整工作流包括数据、基线、训练、验证和误差分析。

## 逐步实现

按顺序运行下面的代码，并在每一步检查 shape、数值范围和中间结果。

In [ ]:
from pathlib import Path
import sys

course_dir = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd() / "07-deep-learning/fundamentals"
sys.path.insert(0, str(course_dir.resolve()))

import numpy as np
import matplotlib.pyplot as plt
from from_scratch import MLP, Momentum, accuracy, make_spiral, train_classifier, train_validation_split

x, y = make_spiral(samples_per_class=100, noise=0.18, seed=21)
x_train, x_valid, y_train, y_valid = train_validation_split(x, y, validation_fraction=0.2, seed=21)
plt.figure(figsize=(5, 4)); plt.scatter(x[:, 0], x[:, 1], c=y, s=14, cmap="viridis"); plt.title("Spiral data"); plt.show()

In [ ]:
model = MLP(2, [64, 64], 3, dropout=0.05, seed=21)
optimizer = Momentum(model.parameters(), learning_rate=0.07, momentum=0.9, weight_decay=1e-4)
history = train_classifier(model, x_train, y_train, optimizer, epochs=220, batch_size=32, seed=21)
train_accuracy = accuracy(model.predict(x_train), y_train)
valid_accuracy = accuracy(model.predict(x_valid), y_valid)
print(f"train={train_accuracy:.3f}, validation={valid_accuracy:.3f}")

In [ ]:
padding = 0.1
x0, x1 = np.meshgrid(
    np.linspace(x[:, 0].min() - padding, x[:, 0].max() + padding, 220),
    np.linspace(x[:, 1].min() - padding, x[:, 1].max() + padding, 220),
)
grid = np.column_stack((x0.ravel(), x1.ravel()))
regions = model.predict(grid).reshape(x0.shape)
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].plot(history["loss"], label="train loss"); axes[0].legend()
axes[1].contourf(x0, x1, regions, alpha=0.35, cmap="viridis")
axes[1].scatter(x_valid[:, 0], x_valid[:, 1], c=y_valid, s=18, cmap="viridis", edgecolors="k")
axes[1].set_title("Validation decision boundary")
plt.tight_layout(); plt.show()

## 检查点

模型只看到训练集；验证准确率评价泛化；固定 seed 使数据、初始化、批次和 Dropout 序列可复现。

## 试一试

移除所有隐藏层或缩小网络，比较决策边界；再调整噪声、正则化和学习率。

## 常见错误

根据验证结果反复修改后仍把它称为测试集；只报告训练准确率；没有固定随机种子却比较微小差异。